<a href="https://colab.research.google.com/github/sadineniManushree/flyrank--internship__ml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sadineniManushree/flyrank--internship__ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

For every query, I figure out what CTR it should get based on its position (top spots normally get way more clicks). Then I compare that to what CTR it's actually getting. If a query has a lot of impressions (people are seeing it) but its real CTR is a lot lower than expected for its position, that's a missed opportunity — people are seeing it but not clicking, even though they should be. The bigger that gap, and the more impressions behind it, the higher the score — because fixing it (better title, meta description, etc.) could win a lot of extra clicks

In [22]:
CTR_BELOW_EXPECTED = 'CTR_BELOW_EXPECTED'

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [24]:
import os

# Make sure the output folder exists
os.makedirs('work/outputs', exist_ok=True)

# 1. Score = (expected CTR for that position - actual CTR) x impressions
df['score'] = (df.groupby('position_bucket')['ctr'].transform('mean') - df['ctr']) * df['impressions']

# 2. Reason code - same label for every flagged row
df['reason_code'] = 'CTR_BELOW_EXPECTED'

# 3. Action label - top 25% of scores get FIX_CTR, rest get MONITOR
threshold = df['score'].quantile(0.75)
df['action'] = df['score'].apply(lambda s: 'FIX_CTR' if s > threshold else 'MONITOR')

# 4. Rank everything - highest score (biggest opportunity) first
df_ranked = df.sort_values('score', ascending=False).reset_index(drop=True)

# 5. Write the CSV
df_ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)

print("Saved", len(df_ranked), "rows to work/outputs/baseline_action_score.csv")
print("Threshold used for FIX_CTR:", round(threshold, 2))
df_ranked.head(10)

Saved 6 rows to work/outputs/baseline_action_score.csv
Threshold used for FIX_CTR: 8.13


,position_bucket,ctr,impressions,score,reason_code,action
0,top,0.030,8000,80.0,CTR_BELOW_EXPECTED,FIX_CTR
1,middle,0.015,4000,10.0,CTR_BELOW_EXPECTED,FIX_CTR
2,bottom,0.005,1000,2.5,CTR_BELOW_EXPECTED,MONITOR
3,bottom,0.010,2000,-5.0,CTR_BELOW_EXPECTED,MONITOR
4,middle,0.020,5000,-12.5,CTR_BELOW_EXPECTED,MONITOR
5,top,0.050,10000,-100.0,CTR_BELOW_EXPECTED,MONITOR


In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [26]:
print(len(df_ranked))

6


In [27]:
print(len(df))

6



## Top Rows Review

Note: The starter dataset only contains 6 rows total, so this reviews
all 6 available rows rather than a top 20.

1. FIX_CTR | CTR_BELOW_EXPECTED | HIGH confidence (8,000 impressions, score 80.0) |
   Wrong if: this "top" position page is new and CTR just hasn't caught up to
   its ranking yet — the gap could shrink on its own with more time.

2. FIX_CTR | CTR_BELOW_EXPECTED | MEDIUM confidence (4,000 impressions, score 10.0) |
   Wrong if: the query intent doesn't match the page content, so a lower CTR
   is expected and not actually a fixable problem.

3. MONITOR | CTR_BELOW_EXPECTED | LOW confidence (1,000 impressions, score 2.5) |
   Wrong if: 1,000 impressions is too small a sample to trust the CTR gap —
   it could just be noise, not a real pattern.

4. MONITOR | CTR_BELOW_EXPECTED | LOW confidence (2,000 impressions, score -5.0) |
   Wrong if: this page is actually doing fine (CTR close to or above
   expected) — the negative score means it's not really a problem, just
   correctly not flagged.

5. MONITOR | CTR_BELOW_EXPECTED | MEDIUM confidence (5,000 impressions, score -12.5) |
   Wrong if: same as above — negative score means CTR here is close to or
   better than expected, so MONITOR is the right call, not a missed opportunity.

6. MONITOR | CTR_BELOW_EXPECTED | HIGH confidence (10,000 impressions, score -100.0) |
   Wrong if: this "top" page actually has a CTR of 0.050 — quite strong — so
   the very negative score correctly shows it's outperforming expectations,
   not underperforming. This row is a good example of the rule correctly
   NOT flagging a healthy page.

In [28]:
top20 = df_ranked.head(20)[['position_bucket', 'ctr', 'impressions', 'score', 'reason_code', 'action']]
print(top20.to_string(index=False))

position_bucket   ctr  impressions  score        reason_code  action
            top 0.030         8000   80.0 CTR_BELOW_EXPECTED FIX_CTR
         middle 0.015         4000   10.0 CTR_BELOW_EXPECTED FIX_CTR
         bottom 0.005         1000    2.5 CTR_BELOW_EXPECTED MONITOR
         bottom 0.010         2000   -5.0 CTR_BELOW_EXPECTED MONITOR
         middle 0.020         5000  -12.5 CTR_BELOW_EXPECTED MONITOR
            top 0.050        10000 -100.0 CTR_BELOW_EXPECTED MONITOR


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Row 2 (middle bucket, ctr 0.015, impressions 4,000, score 10.0, FIX_CTR) is
my weakest pick. The score is small and sits close to the threshold that
separates FIX_CTR from MONITOR, so a tiny change in the data — or a slightly
different threshold choice — could easily flip this row to MONITOR instead.
I'd treat this as a low-priority, low-confidence flag rather than a strong
opportunity, and it's a good candidate to double check manually before
acting on it.

## Leakage Check

I reviewed the code that builds `score`, `reason_code`, and `action`. It uses
only three raw, observable columns: `position_bucket`, `ctr`, and
`impressions`. It does not reference any FlyRank product-decision columns
(health_score, needs_ctr_fix, is_quick_win, priority_score, action_type), and
it does not use any future/outcome data. The score is calculated purely from
current-state signals available at decision time, so there is no circular
reasoning or leakage into this baseline rule.

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.